In [1]:
import pandas as pd, numpy as np

d = pd.read_csv("nifty50_features.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
d["gap_pct"]      = (d["Open"].shift(-1) / d["Close"] - 1) * 100            # next open vs today's close
d["intraday_pct"] = (d["Close"].shift(-1) / d["Open"].shift(-1) - 1) * 100   # next close vs next open
d["next_ret_pct"] = (d["Close"].shift(-1) / d["Close"] - 1) * 100            # whole next day
d = d.dropna(subset=["gap_pct", "intraday_pct"]).reset_index(drop=True)

drivers = ["sp500_chg", "usdinr_chg", "crude_chg", "vix_chg"]
print(d[drivers + ["gap_pct", "intraday_pct", "next_ret_pct"]].corr().round(2)
        .loc[drivers, ["gap_pct", "intraday_pct", "next_ret_pct"]])

            gap_pct  intraday_pct  next_ret_pct
sp500_chg      0.45         -0.02          0.25
usdinr_chg     0.03         -0.00          0.01
crude_chg      0.12          0.02          0.09
vix_chg       -0.04          0.02         -0.01


In [3]:
slope, intercept = np.polyfit(d["sp500_chg"], d["gap_pct"], 1)
r = d["sp500_chg"].corr(d["gap_pct"])
print(f"gap ≈ {intercept:.3f}% + {slope:.2f} × (S&P 500 move)   | correlation {r:.2f} | R² {r*r:.2f}")

print(d.groupby(d["Date"].dt.year).apply(lambda g: pd.Series({
    "days": len(g),
    "corr": g["sp500_chg"].corr(g["gap_pct"]),
    "NIFTY gap per 1% S&P": np.polyfit(g["sp500_chg"], g["gap_pct"], 1)[0],
})).round(2))

gap ≈ 0.089% + 0.25 × (S&P 500 move)   | correlation 0.45 | R² 0.20
       days  corr  NIFTY gap per 1% S&P
Date                                   
2010  201.0  0.09                  0.02
2011  242.0  0.66                  0.35
2012  242.0  0.58                  0.37
2013  247.0  0.46                  0.35
2014  240.0  0.56                  0.30
2015  244.0  0.48                  0.24
2016  244.0  0.31                  0.23
2017  248.0  0.40                  0.30
2018  245.0  0.58                  0.28
2019  241.0  0.13                  0.06
2020  250.0  0.35                  0.22
2021  248.0  0.45                  0.28
2022  248.0  0.68                  0.33
2023  245.0  0.55                  0.22
2024  246.0  0.45                  0.26
2025  249.0  0.44                  0.19
2026  190.0  0.34                  0.30


In [5]:
bins = [-np.inf, -1, -0.3, 0.3, 1, np.inf]
labels = ["S&P < -1%", "-1% to -0.3%", "-0.3% to +0.3%", "+0.3% to +1%", "S&P > +1%"]
d["sp_bucket"] = pd.cut(d["sp500_chg"], bins=bins, labels=labels)
print(d.groupby("sp_bucket", observed=True).agg(
    days=("gap_pct", "size"),
    avg_gap_pct=("gap_pct", "mean"),
    pct_days_gap_up=("gap_pct", lambda s: (s > 0).mean() * 100),
).round(2))

                days  avg_gap_pct  pct_days_gap_up
sp_bucket                                         
S&P < -1%        447        -0.45            25.28
-1% to -0.3%     692        -0.05            43.79
-0.3% to +0.3%  1465         0.10            66.48
+0.3% to +1%     930         0.26            82.37
S&P > +1%        536         0.50            90.30


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

feats = ["sp500_chg", "usdinr_chg", "crude_chg", "vix_chg"]
train, test = d[d["Date"] < "2023-01-01"], d[d["Date"] >= "2023-01-01"]

m = LinearRegression().fit(train[feats], train["gap_pct"])
p = m.predict(test[feats])
y = test["gap_pct"]

print("Model            MAE %", round(mean_absolute_error(y, p), 3), "| R²", round(r2_score(y, p), 3))
print("Predict 'no gap' MAE %", round(mean_absolute_error(y, np.zeros(len(y))), 3))
print("Direction right:", round((np.sign(p) == np.sign(y)).mean() * 100, 1), "% of days")
print("Coefficients:", dict(zip(feats, m.coef_.round(3))))

resid_sd = float(np.std(y - p))
print("Typical error (1 SD) %:", round(resid_sd, 3))

Model            MAE % 0.287 | R² 0.149
Predict 'no gap' MAE % 0.33
Direction right: 70.1 % of days
Coefficients: {'sp500_chg': 0.252, 'usdinr_chg': 0.029, 'crude_chg': 0.011, 'vix_chg': 0.004}
Typical error (1 SD) %: 0.483
